# RSNA Knee — EDA & Image Quality Check
Runs on TACC (kernel: **Python (rsna_env)**). Data: `/scratch/07880/devansh/rsna_knee/data`, code/results: `/work/07880/devansh/lonestar/rsna_knee`.

Sections: 1) cohort overview · 2) teacher labels · 3) series inventory · 4) image QC montages · 5) intensity / blank-slice checks · 6) raw DICOM vs preprocessed · 7) preprocessing errors · 8) training progress

In [ ]:
from pathlib import Path
import json, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

random.seed(42); np.random.seed(42)
WORK = Path("/work/07880/devansh/lonestar/rsna_knee")
DATA = Path("/scratch/07880/devansh/rsna_knee/data")
PROC = DATA/"proc/train"
RAW  = DATA/"raw/train_series"

LABELS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
          "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
train  = pd.read_csv(DATA/"csv/train.csv")
series = pd.read_csv(DATA/"csv/train_series.csv")
folds  = pd.read_csv(DATA/"csv/train_folds.csv")
print(f"{len(train)} studies | {len(series)} series | folds file: {len(folds)} rows")

## 1. Cohort overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
folds["lang"].value_counts().plot.bar(ax=axes[0], title="Report language (≈ site proxy)")
series.groupby("StudyInstanceUID").size().hist(bins=range(2, 16), ax=axes[1])
axes[1].set_title("Series per study")
folds["fold"].value_counts().sort_index().plot.bar(ax=axes[2], title="Fold sizes (-1 = gold-58 holdout)")
plt.tight_layout(); plt.show()

In [ ]:
# gold label prevalence vs teacher-derived prevalence on all studies
gold = train.dropna(subset=LABELS)
prev = pd.DataFrame({
    "gold_58": gold[LABELS].mean(),
    "teacher@0.5_all": [(folds[f"t_{l}"] > 0.5).mean() for l in LABELS],
})
prev.plot.bar(figsize=(12, 3.5), title="Label prevalence: gold-58 vs teacher labels (all 4,407)")
plt.axhline(0, c="k"); plt.tight_layout(); plt.show()
prev.round(3)

## 2. Teacher soft labels — score distributions
Well-behaved teacher scores should be bimodal (confident 0s and 1s). Heavy mass in the middle = uncertain labels; those studies contribute weak gradients.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
for ax, lab in zip(axes.ravel(), LABELS):
    ax.hist(folds[f"t_{lab}"], bins=40)
    ax.set_title(lab, fontsize=9); ax.set_yscale("log")
plt.suptitle("Teacher ensemble soft scores (log y)"); plt.tight_layout(); plt.show()
mid = ((folds[[f"t_{l}" for l in LABELS]] > 0.2) & (folds[[f"t_{l}" for l in LABELS]] < 0.8)).mean()
print("fraction of scores in the uncertain 0.2-0.8 band:"); print(mid.round(3).to_string())

## 3. Series inventory

In [ ]:
print(series.groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]).size()
      .rename("n_series").to_string())
meta = pd.read_csv(PROC/"preprocess_meta.csv")
ok = meta[meta.status.isin(["ok", "cached"])]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
ok["n_slices"].clip(0, 120).hist(bins=60, ax=axes[0]); axes[0].set_title("Slices per series")
if "raw_h" in ok:
    sizes = (ok["raw_h"].astype("Int64").astype(str) + "x" + ok["raw_w"].astype("Int64").astype(str))
    sizes.value_counts().head(12).plot.bar(ax=axes[1]); axes[1].set_title("Raw in-plane sizes (top 12)")
meta["status"].value_counts().plot.bar(ax=axes[2]); axes[2].set_title("Preprocess status")
plt.tight_layout(); plt.show()

## 4. Image QC — study montage
Re-run this cell to sample a new study. Sanity-check: anatomy looks like a knee, planes match the metadata, contrast reasonable, no corrupted frames.

In [ ]:
uid = random.choice(folds.StudyInstanceUID.tolist())
rows = series[series.StudyInstanceUID == uid]
teacher = folds.set_index("StudyInstanceUID").loc[uid, [f"t_{l}" for l in LABELS]]
print("study", uid)
print("teacher>0.5:", [l for l in LABELS if teacher[f"t_{l}"] > 0.5] or "none")
fig, axes = plt.subplots(1, len(rows), figsize=(3.2*len(rows), 3.5))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, rows.itertuples()):
    p = PROC/uid/f"{r.SeriesInstanceUID}.npy"
    if not p.exists():
        ax.set_title("missing"); ax.axis("off"); continue
    v = np.load(p, mmap_mode="r")
    ax.imshow(v[v.shape[0]//2], cmap="gray")
    ax.set_title(f"{r.Anatomical_Plane}\nfluid={r.Fluid_Sensitive} fatsup={r.Fat_Suppression}\n{v.shape[0]} slices", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

### Slice sweep through one series

In [ ]:
r = rows.iloc[0]
v = np.load(PROC/uid/f"{r.SeriesInstanceUID}.npy")
idx = np.linspace(0, v.shape[0]-1, 12).astype(int)
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(v[i], cmap="gray"); ax.set_title(f"slice {i}", fontsize=8); ax.axis("off")
plt.suptitle(f"{r.Anatomical_Plane} series, {v.shape[0]} slices"); plt.tight_layout(); plt.show()

### Random per-plane grid (12 random series per plane)

In [ ]:
for plane in ["Sagittal", "Coronal", "Axial"]:
    sub = series[series.Anatomical_Plane == plane].sample(12, random_state=None)
    fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
    for ax, r in zip(axes.ravel(), sub.itertuples()):
        p = PROC/r.StudyInstanceUID/f"{r.SeriesInstanceUID}.npy"
        ax.axis("off")
        if p.exists():
            v = np.load(p, mmap_mode="r"); ax.imshow(v[v.shape[0]//2], cmap="gray")
    plt.suptitle(plane); plt.tight_layout(); plt.show()

## 5. Intensity & blank-slice checks
Preprocessing clips at the 1st/99th percentile then scales to 0-255 — so ~1% at each extreme is expected. Series with much higher saturation, or many near-constant slices, deserve a look.

In [ ]:
sample = series.sample(300, random_state=42)
stats = []
for r in sample.itertuples():
    p = PROC/r.StudyInstanceUID/f"{r.SeriesInstanceUID}.npy"
    if not p.exists(): continue
    v = np.load(p)
    slice_std = v.reshape(v.shape[0], -1).std(1)
    stats.append(dict(uid=r.StudyInstanceUID, sid=r.SeriesInstanceUID, plane=r.Anatomical_Plane,
                      n=v.shape[0], sat0=(v == 0).mean(), sat255=(v == 255).mean(),
                      mean=v.mean(), frac_blank=(slice_std < 5).mean()))
qc = pd.DataFrame(stats)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
qc["sat255"].hist(bins=40, ax=axes[0]); axes[0].set_title("fraction of pixels at 255")
qc["mean"].hist(bins=40, ax=axes[1]);   axes[1].set_title("volume mean intensity")
qc["frac_blank"].hist(bins=40, ax=axes[2]); axes[2].set_title("fraction near-blank slices (std<5)")
plt.tight_layout(); plt.show()
flag = qc[(qc.frac_blank > 0.2) | (qc.sat255 > 0.10)]
print(f"flagged {len(flag)}/{len(qc)} sampled series"); flag.head(10)

In [ ]:
# look at the worst flagged series (if any)
if len(flag):
    r = flag.sort_values("frac_blank", ascending=False).iloc[0]
    v = np.load(PROC/r.uid/f"{r.sid}.npy")
    idx = np.linspace(0, v.shape[0]-1, 12).astype(int)
    fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
    for ax, i in zip(axes.ravel(), idx):
        ax.imshow(v[i], cmap="gray"); ax.set_title(f"slice {i} std={v[i].std():.0f}", fontsize=8); ax.axis("off")
    plt.suptitle(f"most-flagged series ({r.plane}, frac_blank={r.frac_blank:.2f})"); plt.tight_layout(); plt.show()

## 6. Raw DICOM vs preprocessed
Confirms sorting/orientation/windowing didn't mangle anything.

In [ ]:
import pydicom
r = series.sample(1, random_state=None).iloc[0]
sdir = RAW/r.StudyInstanceUID/r.SeriesInstanceUID
dcms = sorted(sdir.glob("*.dcm"))
ds = pydicom.dcmread(str(dcms[len(dcms)//2]))
raw_img = ds.pixel_array.astype(float)
v = np.load(PROC/r.StudyInstanceUID/f"{r.SeriesInstanceUID}.npy")
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(raw_img, cmap="gray"); axes[0].set_title(f"raw middle file {raw_img.shape}\n{ds.file_meta.TransferSyntaxUID.name}", fontsize=8)
axes[1].imshow(v[v.shape[0]//2], cmap="gray"); axes[1].set_title(f"preprocessed mid-slice {v.shape[1:]}" , fontsize=8)
axes[2].hist(raw_img.ravel(), bins=60); axes[2].set_title("raw intensity histogram", fontsize=8)
for ax in axes[:2]: ax.axis("off")
plt.tight_layout(); plt.show()
print({k: getattr(ds, k, "?") for k in ["Modality","MagneticFieldStrength","Manufacturer","SliceThickness","PixelSpacing"]})

## 7. Preprocessing errors

In [ ]:
err = meta[meta.status == "error"]
print(f"{len(err)} error series")
for _, e in err.iterrows():
    print("\n---", e.StudyInstanceUID, e.SeriesInstanceUID)
    print(str(e.get("error"))[:400])

## 8. Training progress (live)
Parses the fold logs on the node — re-run any time.

In [ ]:
import re
hist = {}
for f in range(4):
    p = WORK/f"logs/train_fold{f}.log"
    if not p.exists(): continue
    rows = re.findall(r"ep(\d+): loss ([\d.]+) \| val-teacher AUC ([\d.]+) \| gold AUC ([\d.]+)", p.read_text())
    if rows:
        hist[f] = pd.DataFrame(rows, columns=["ep","loss","val_auc","gold_auc"]).astype(float)
if hist:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for f, h in hist.items():
        axes[0].plot(h.ep, h.val_auc, marker="o", label=f"fold {f}")
        axes[1].plot(h.ep, h.gold_auc, marker="o", label=f"fold {f}")
    axes[0].set_title("val AUC vs teacher labels"); axes[1].set_title("AUC vs gold-58 (honest)")
    for ax in axes: ax.legend(); ax.grid(alpha=.3); ax.set_xlabel("epoch")
    plt.tight_layout(); plt.show()
else:
    print("no epoch lines in logs yet")